# Day 33 — Linear algebra & matrices for ML intuition
Objectives:
- Vectors/matrices with NumPy.
- Normal equation for linear regression (closed form).
- Geometric view: projections, rank (intuition).

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
# Synthetic linear regression y = Xw + noise
n, d = 200, 3
X = rng.normal(size=(n,d))
true_w = np.array([2.0,-1.0,0.5])
y = X @ true_w + rng.normal(scale=0.5, size=n)
# Closed-form (normal equation) solution: w = (X^T X)^{-1} X^T y
w_hat = np.linalg.pinv(X.T @ X) @ X.T @ y
true_w, w_hat


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — matrix shapes, linear transformations, rank, and stable least squares

### Mental model

A vector or matrix is both stored numbers and a transformation with a
shape contract. For `A @ B`, the inner dimensions must agree; the outer
dimensions determine the result. In a design matrix, rows represent
observations and columns represent features. Coefficients map feature
space to predictions.

Least squares chooses coefficients that minimize squared residuals.
Full column rank gives a unique coefficient solution, but near
collinearity can make that solution numerically sensitive. Prediction
may remain stable even while individual coefficients swing, which is
why rank and condition number belong in the diagnostic story.

### Read the API before running it

- **`A.shape` and `B.shape`:** state the dimensional contract before multiplication; never infer it from a successful broadcast.
- **`A @ B`:** performs matrix multiplication, which is different from elementwise `A * B`.
- **`np.linalg.lstsq(X, y, rcond=None)`:** solves least squares directly and reports rank without explicitly forming an unstable inverse.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — trace a matrix product by shape and by hand

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** Columns of the matrix and positions in the coefficient vector use the same feature order.

In [ ]:
import numpy as np

observations = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
coefficients = np.array([10.0, -1.0])
predictions = observations @ coefficients
print(observations.shape, coefficients.shape, predictions.shape)
print(predictions)
assert np.array_equal(predictions, np.array([8.0, 26.0, 44.0]))

**Expected observation:** A `(3, 2)` matrix multiplied by a `(2,)` vector returns one prediction for each of three rows.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — observe coefficient instability under collinearity

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** The nearly duplicated columns do not represent independently identifiable effects.

In [ ]:
import numpy as np

x = np.linspace(0.0, 1.0, 50)
X = np.column_stack([np.ones_like(x), x, x + 1e-10 * np.arange(x.size)])
y = 3.0 + 2.0 * x

coef, residuals, rank, singular_values = np.linalg.lstsq(X, y, rcond=None)
condition = np.linalg.cond(X)
max_prediction_error = np.max(np.abs(X @ coef - y))
print({"rank": rank, "condition": condition,
       "coef": coef, "prediction_error": max_prediction_error})

**Expected observation:** The condition number is huge and individual duplicate-feature coefficients are not trustworthy even though predictions are accurate.

### Debugging and practice ramp

**Common mistake:** Computing `(X.T @ X) ** -1` or `np.linalg.inv(X.T @ X)` as a default least-squares recipe.

**Diagnostic:** Print shapes, rank, singular values, and condition number; compare `lstsq` predictions with the target before interpreting coefficients.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define matrix shapes, linear transformations, rank, and stable least squares in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not assign meaning to a coefficient when feature order, units, rank, or intercept handling is unknown.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Add an intercept column of ones and recompute the coefficients.

**Verify:** For task `Add an intercept column of ones and recompute the coefficients`, show the formula or intermediate quantities and check the final value independently rather than trusting one library call.






2. Compare the closed-form result with scikit-learn's `LinearRegression` on the
   same data.

**Verify:** For task `Compare the closed-form result with scikit-learn's LinearRegression on the`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






3. Explain when the normal equation becomes numerically unstable and why
   iterative methods are often used for larger problems.

**Verify:** For task `Explain when the normal equation becomes numerically unstable and why`, state one precise claim, the evidence supporting it, the governing assumption, and a counterexample or limitation.







### Progressive hints

1. The augmented matrix has one more column. Decide whether the first or last
   coefficient will represent the intercept before inspecting the result.
2. `LinearRegression(fit_intercept=True)` keeps the intercept separate from
   `coef_`; align the two representations before comparing.
3. Construct two almost-duplicate feature columns and inspect
   `np.linalg.cond(X)` or the singular values. Think about both memory and
   computational complexity as dimensions grow.

### Additional mastery practice

Treat shapes, rank, and conditioning as part of every matrix contract. Prefer stable solvers to symbolic formulas that require an explicit inverse.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Shape tracing:** For X with shape (120, 8), beta with shape (8,), and y with shape (120,), trace the shapes of X.T, X.T @ X, X @ beta, and residuals. Then explain what changes if beta is shaped (8, 1).
   **Progressive hint:** Write shapes beside every operand before multiplying. A column vector preserves a trailing dimension that can trigger broadcasting.

**Verify:** For task `Shape tracing: For X with shape (120, 8), beta with shape (8,), and y with shape (120,), trac...`, state one precise claim, the evidence supporting it, the governing assumption, and a counterexample or limitation.







5. **Rank-deficiency debugging:** Construct a design matrix whose third column equals the sum of the first two. Compare `np.linalg.solve(X.T @ X, X.T @ y)` with `np.linalg.lstsq(X, y, rcond=None)` and interpret the rank.
   **Progressive hint:** The dependent column makes X.T @ X singular. `lstsq` returns a minimum-norm solution plus rank information without forming an inverse.

**Verify:** For task `Rank-deficiency debugging: Construct a design matrix whose third column equals the sum of the...`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed; then reproduce the failure first, capture its smallest observable symptom, apply one scoped fix, and rerun the failing plus normal case.







6. **Robust vector operation:** Implement cosine similarity for two one-dimensional vectors. Validate equal shapes and define behavior for a zero vector.
   **Progressive hint:** Compute dot(a,b)/(norm(a)*norm(b)); a zero norm makes the angle undefined, so do not quietly add an epsilon without documenting it.

**Verify:** For task `Robust vector operation: Implement cosine similarity for two one-dimensional vectors. Validat...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then produce the requested artifact with every named field/control and walk one allowed plus one rejected scenario through it.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Shape tracing


# Practice 5 — Rank-deficiency debugging


# Practice 6 — Robust vector operation
